# 13 端侧模型选型（可运行决策器）

把「场景约束 → 候选模型 → 量化后体积/速度/能力」收成打分器，避免拍脑袋选型。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
import time
import hashlib
import hmac
import json
import math

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__}")

## 13.1 模型目录（2026 示意表）

In [ ]:
CATALOG = [
    # name, params_b, ctx_k, zh, agent, code, multimodal, notes
    ("Gemma4-E2B", 2.0, 8, 0.7, 0.7, 0.6, 0.5, "Android 友好, QAT INT4"),
    ("Gemma4-E4B", 4.0, 8, 0.75, 0.95, 0.7, 0.6, "Agent 原生"),
    ("Qwen3-4B", 4.0, 32, 0.98, 0.85, 0.8, 0.5, "中文首选"),
    ("Qwen3-8B", 8.0, 128, 0.97, 0.88, 0.85, 0.5, "长上下文+MLA"),
    ("MiniCPM3", 4.0, 32, 0.92, 0.7, 0.65, 0.95, "多模态强"),
    ("Llama3.2-1B", 1.0, 128, 0.55, 0.45, 0.5, 0.3, "低端设备"),
    ("Llama3.2-3B", 3.0, 128, 0.65, 0.6, 0.65, 0.35, "生态成熟"),
    ("Phi-4-mini", 3.8, 16, 0.7, 0.75, 0.8, 0.4, "推理强"),
    ("DS-Coder-Lite", 2.4, 128, 0.6, 0.55, 0.98, 0.2, "代码; MoE激活"),
    ("HY-1.8B-2Bit", 1.8, 4, 0.8, 0.4, 0.45, 0.2, "极致压缩"),
]


def size_mb(params_b, bits=4):
    return params_b * 1e9 * bits / 8 / 1e6

## 13.2 场景约束打分

In [ ]:
@dataclass
class Constraints:
    mem_mb: float = 2048
    need_zh: float = 0.5
    need_agent: float = 0.3
    need_code: float = 0.2
    need_mm: float = 0.0
    min_ctx_k: int = 4
    bits: int = 4


def score_model(row, c: Constraints):
    name, pb, ctx, zh, agent, code, mm, notes = row
    sz = size_mb(pb, c.bits)
    if sz > c.mem_mb * 0.85 or ctx < c.min_ctx_k:
        return None
    # 硬门槛：场景强需求时过滤明显不合格模型
    if c.need_zh >= 0.8 and zh < 0.75:
        return None
    if c.need_agent >= 0.8 and agent < 0.7:
        return None
    if c.need_code >= 0.8 and code < 0.75:
        return None
    if c.need_mm >= 0.8 and mm < 0.7:
        return None
    denom = max(c.need_zh + c.need_agent + c.need_code + c.need_mm, 1e-6)
    capability = (
        c.need_zh * zh + c.need_agent * agent + c.need_code * code + c.need_mm * mm
    ) / denom
    speed = 1.5 / pb
    # 能力优先，体积/速度为辅
    total = 0.72 * capability + 0.15 * speed + 0.13 * (1 - sz / c.mem_mb)
    return {
        "name": name,
        "score": round(total, 3),
        "size_mb": round(sz, 0),
        "ctx_k": ctx,
        "notes": notes,
        "capability": round(capability, 3),
    }


def rank(c: Constraints, topk=5):
    scored = [score_model(r, c) for r in CATALOG]
    scored = [s for s in scored if s]
    return sorted(scored, key=lambda x: -x["score"])[:topk]


## 13.3 场景演示

In [ ]:
scenarios = {
    "中文手机助手 3GB 可用": Constraints(mem_mb=3000, need_zh=1.0, need_agent=0.6, need_code=0.2, min_ctx_k=8),
    "低端机 1.2GB": Constraints(mem_mb=1200, need_zh=0.7, need_agent=0.2, bits=2, min_ctx_k=4),
    "端侧 Agent 工具调用": Constraints(mem_mb=3500, need_agent=1.0, need_zh=0.6, need_code=0.4, min_ctx_k=8),
    "离线看图问答": Constraints(mem_mb=3500, need_mm=1.0, need_zh=0.8, need_agent=0.3, min_ctx_k=8),
    "代码补全插件": Constraints(mem_mb=2500, need_code=1.0, need_zh=0.3, min_ctx_k=32),
}

for title, cons in scenarios.items():
    print(f"\n=== {title} ===")
    for i, row in enumerate(rank(cons), 1):
        print(f"{i}. {row['name']:14s} score={row['score']:.3f}  {row['size_mb']:.0f}MB  ctx={row['ctx_k']}K  {row['notes']}")

## 小结

1. 先卡 **内存与上下文**，再比能力。
2. 中文优先 Qwen/MiniCPM；Agent 看 Gemma E4B；低端看 1B/2-bit。
3. 表内分数是教学相对值，上线前仍要用第 12 章指标实测。